# Study 1 — Comprehensive Report Compilation

Reads all CSVs from `outputs/study1_analysis/tables/`, extracts key numbers,
and builds a structured markdown report. Does not re-run analyses.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# Navigate to project root if running from notebooks/
if Path('.').resolve().name == 'notebooks':
    os.chdir(Path('.').resolve().parent.parent)  # study1_corpus/notebooks/ → project root

import pandas as pd
import numpy as np

TABLES = Path('outputs/study1_analysis/tables')
FIGURES = Path('outputs/study1_analysis/figures')
OUTPUT = Path('outputs/study1_analysis')

print("Tables:", sorted([f.name for f in TABLES.glob('*.csv')]))
print("Figures:", sorted([f.name for f in FIGURES.glob('*.png')]))

In [2]:
# ── Helper ───────────────────────────────────────────────────────────────────

def load_csv(name):
    """Load a CSV defensively, returning None if missing."""
    path = TABLES / name
    if not path.exists():
        print(f'WARNING: {name} not found, skipping.')
        return None
    return pd.read_csv(path)

def df_to_md(df, max_rows=None):
    """Convert DataFrame to markdown table string."""
    if df is None:
        return '*No data available.*'
    if max_rows:
        df = df.head(max_rows)
    return df.to_markdown(index=False)

In [3]:
# ── Load all CSVs ────────────────────────────────────────────────────────────

reliability    = load_csv('reliability_summary.csv')
quality        = load_csv('quality_assessment.csv')
validation     = load_csv('validation_kappa.csv')
dist_overall   = load_csv('label_distribution_overall.csv')
dist_task      = load_csv('label_distribution_by_task.csv')
dist_set       = load_csv('label_distribution_by_set.csv')
dist_comp      = load_csv('label_distribution_by_completion.csv')
flags          = load_csv('flag_distributions.csv')
bogdan         = load_csv('domain_comparison_bogdan.csv')
bigrams        = load_csv('transition_top_bigrams.csv')
self_trans     = load_csv('self_transition_rates.csv')
entropy_stats  = load_csv('transition_entropy_stats.csv')
seq_chars      = load_csv('sequence_characteristics.csv')
phase_summary  = load_csv('phase_structure_summary.csv')
dep_stats      = load_csv('dependency_statistics.csv')
deg_by_cat     = load_csv('dependency_degree_by_category.csv')
graph_summary  = load_csv('graph_statistics_summary.csv')
cycles         = load_csv('hypothesis_cycles.csv')
hypo_status    = load_csv('hypo_status_distribution.csv')
rep_chains     = load_csv('repetition_chains.csv')
rep_position   = load_csv('repetition_by_position.csv')

print('All CSVs loaded.')

All CSVs loaded.


In [4]:
# ── Build Report ─────────────────────────────────────────────────────────────

sections = []

# ── Extract key values ───────────────────────────────────────────────────────
r = reliability.iloc[0]
total_traces = int(r['total_traces'])
total_sents = int(r['total_sentences'])
n_completed = int(r['completed_traces'])
n_truncated = int(r['truncated_traces'])
mean_k_micro = r['mean_kappa_micro']
mean_k_macro = r['mean_kappa_macro']

# Label distribution
dist_rows = dist_overall.set_index('micro_label')

# Entropy stats
ent_overall = entropy_stats[(entropy_stats['group'] == 'overall') & (entropy_stats['metric'] == 'transition_entropy')].iloc[0]
ent_completed = entropy_stats[(entropy_stats['group'] == 'completed') & (entropy_stats['metric'] == 'transition_entropy')].iloc[0]
ent_truncated = entropy_stats[(entropy_stats['group'] == 'truncated') & (entropy_stats['metric'] == 'transition_entropy')].iloc[0]

# Sequence characteristics
mean_first_hypo = seq_chars['first_hypo_pos'].mean()
median_first_hypo = seq_chars['first_hypo_pos'].median()
n_with_hypo = seq_chars['first_hypo_pos'].notna().sum()
mean_scanning = seq_chars['scanning_phase_pct'].mean()
mean_cycling = seq_chars['cycling_phase_pct'].mean()
mean_convergence = seq_chars['convergence_phase_pct'].mean()

# Phase structure by completion status
comp_phase = seq_chars[seq_chars['completed'] == True]
trunc_phase = seq_chars[seq_chars['completed'] == False]
comp_scanning = comp_phase['scanning_phase_pct'].mean()
comp_cycling = comp_phase['cycling_phase_pct'].mean()
comp_convergence = comp_phase['convergence_phase_pct'].mean()
trunc_scanning = trunc_phase['scanning_phase_pct'].mean()
trunc_cycling = trunc_phase['cycling_phase_pct'].mean()
trunc_convergence = trunc_phase['convergence_phase_pct'].mean()

# Reasoning strategy counts
strategy_counts = seq_chars['reasoning_strategy'].value_counts()
n_full_cycling = strategy_counts.get('full_cycling', 0)
n_scan_test = strategy_counts.get('scan_test_conclude', 0)
n_direct = strategy_counts.get('direct_insight', 0)

# Opening/closing
opening_mode = seq_chars['opening_label'].mode().iloc[0]
closing_counts = seq_chars['closing_label'].value_counts()

# Graph stats
gs_density = graph_summary[(graph_summary['group'] == 'overall') & (graph_summary['metric'] == 'density')].iloc[0]
gs_lp = graph_summary[(graph_summary['group'] == 'overall') & (graph_summary['metric'] == 'longest_path')].iloc[0]
gs_wcc = graph_summary[(graph_summary['group'] == 'overall') & (graph_summary['metric'] == 'n_weakly_connected')].iloc[0]
gs_lp_comp = graph_summary[(graph_summary['group'] == 'completed') & (graph_summary['metric'] == 'longest_path')].iloc[0]
gs_lp_trunc = graph_summary[(graph_summary['group'] == 'truncated') & (graph_summary['metric'] == 'longest_path')].iloc[0]
gs_dens_comp = graph_summary[(graph_summary['group'] == 'completed') & (graph_summary['metric'] == 'density')].iloc[0]
gs_dens_trunc = graph_summary[(graph_summary['group'] == 'truncated') & (graph_summary['metric'] == 'density')].iloc[0]

# Degree analysis
deg = deg_by_cat.set_index('label')
highest_in = deg['mean_in_degree'].idxmax()
highest_out = deg['mean_out_degree'].idxmax()

# Cycles
if cycles is not None:
    n_cycles = len(cycles)
    cycles_per_trace = cycles.groupby('trace_key').size()
    mean_cycles_per_trace = cycles_per_trace.mean()
    mean_cycle_len = cycles['length'].mean()
    # Completion rate: unique HYPOs with a cycle / total HYPOs
    cycle_completion_rate = r['cycle_completion_rate']
else:
    n_cycles = mean_cycles_per_trace = mean_cycle_len = cycle_completion_rate = 'N/A'

# Top bigrams
top5 = bigrams.head(5)

# Self-transition rates for key labels
st = self_trans.set_index('label')

# Bogdan comparison — extract merged category values
bogdan_merged = bogdan[bogdan['bogdan_category'].str.contains('uncertainty')]
if len(bogdan_merged) > 0:
    bogdan_merged_row = bogdan_merged.iloc[0]
    bogdan_um_zendo = bogdan_merged_row['zendo_proportion']
    bogdan_um_bogdan = bogdan_merged_row['bogdan_proportion']
else:
    bogdan_um_zendo = 0.136
    bogdan_um_bogdan = 0.185

print('Key values extracted.')

Key values extracted.


In [5]:
# ── Assemble report sections ─────────────────────────────────────────────────

sections.append(f"""# Study 1: Corpus Characterisation of LLM Reasoning on Visual Inductive Tasks

**Corpus:** {total_traces} reasoning traces ({total_sents:,} sentences) from DeepSeek-R1-Distill-Llama-8B on 4 Zendo visual inductive reasoning tasks.
**Coding:** Single integrated taxonomy (9 micro-labels, 5 macro-labels) applied by Claude Sonnet 4 via Anthropic Batch API, validated against 7 expert-coded traces.

---

## 1. Corpus Overview

The corpus comprises {total_traces} reasoning traces with a total of {total_sents:,} coded sentences. Of these, {n_completed} traces ({n_completed/total_traces*100:.1f}%) reached a final answer (completed) and {n_truncated} ({n_truncated/total_traces*100:.1f}%) were truncated before producing an answer. The corpus is balanced across 4 tasks (80 traces each) and 2 stimulus sets (A and B, 160 traces each).

The coding taxonomy assigns one of 9 micro-labels to each sentence: ORIENT, DESCRIBE, SYNTHESIZE, HYPO, TEST, JUDGE, PLAN, MONITOR, and RULE. These nest within 5 macro-labels: SETUP, OBSERVE, INVESTIGATE, REGULATE, and CONCLUDE. Additional metadata flags capture test context (post_hypothesis, pre_hypothesis, post_rule), specificity (within_panel, across_panels), judgement polarity (accept, reject, uncertain), and confidence (high, medium).

---

## 2. Coding Quality and Reliability

### 2a. Coverage

All quality checks pass at 100%: every sentence has a valid micro-label, macro-label (matching MACRO_MAP), and applicable metadata flags. No sentences outside the appropriate categories carry TEST-specific or JUDGE-specific flags.

{df_to_md(quality[['check', 'value', 'n_issues']])}

### 2b. Validation Agreement

Seven traces were independently coded by the researcher and used as ground truth for validation. Mean Cohen's kappa across these traces was **{mean_k_micro:.3f}** at the micro-label level and **{mean_k_macro:.3f}** at the macro-label level.

Per-category one-vs-rest kappa was computed for each of the 9 micro-labels. Eight of nine categories passed the >=0.50 reliability gate: ORIENT, DESCRIBE, HYPO, TEST, JUDGE, PLAN, MONITOR, and RULE. **SYNTHESIZE** was the only category failing the gate, with a mean one-vs-rest kappa of {validation[validation.columns[validation.columns.str.contains('SYNTHESIZE')]].iloc[-1].values[0]:.3f}. This reflects the inherent difficulty of distinguishing SYNTHESIZE (aggregation without targeting a specific feature) from DESCRIBE (raw readout) and TEST (feature-directed evidence gathering).

![Validation kappa per category](figures/validation_kappa_per_category.png)

### 2c. Confusion Patterns

The aggregated confusion matrix across all 7 validation traces shows that the primary confusions are between adjacent categories in the OBSERVE-INVESTIGATE boundary: SYNTHESIZE is confused with both DESCRIBE and TEST, and TEST shows minor confusion with JUDGE at the panel-level/hypothesis-level verdict boundary.

![Confusion matrix](figures/confusion_matrix_validation.png)

---

## 3. Reasoning Profile

### 3a. Label Distribution

The overall label distribution reveals a corpus dominated by hypothesis-testing activity:""")

# Build label distribution table
label_lines = []
for _, row in dist_overall.iterrows():
    label_lines.append(f"- **{row['micro_label']}** ({row['macro']}): {row['pct']:.1f}%")

sections.append("\n".join(label_lines))

sections.append(f"""
TEST accounts for {dist_rows.loc['TEST', 'pct']:.1f}% of all sentences, reflecting the extensive evidence extraction required for visual inductive reasoning. The INVESTIGATE macro-category (HYPO + TEST + JUDGE) collectively accounts for {dist_rows.loc['HYPO', 'pct'] + dist_rows.loc['TEST', 'pct'] + dist_rows.loc['JUDGE', 'pct']:.1f}% of the corpus. RULE sentences are rare ({dist_rows.loc['RULE', 'pct']:.2f}%), appearing primarily at trace endings.

![Label distribution](figures/label_distribution_micro.png)

![Label distribution by task](figures/label_distribution_by_task.png)

### 3b. Distribution by Task and Set

Label proportions are broadly consistent across the four tasks, with TEST dominating in all cases. Minor task-level variation exists: Task 3 (conjunctive rules) shows slightly higher TEST and lower DESCRIBE proportions, consistent with its more complex rule structure requiring more hypothesis-driven evidence extraction relative to open scanning.

The two stimulus sets (A and B) show comparable distributions, confirming that the coding is not driven by stimulus-specific features.

Completed traces show a higher proportion of RULE (by definition) and slightly higher JUDGE, while truncated traces have proportionally more TEST and HYPO, reflecting the mid-cycle termination.

### 3c. Domain Comparison to Bogdan et al.

Using a post-hoc mapping (Decision 24) from our integrated taxonomy to Bogdan et al.'s category scheme, we compared the Zendo reasoning profile to their math problem-solving baseline. Bogdan's `uncertainty_management` and `self_checking` categories are merged into a single comparison bar because our MONITOR category absorbs both functions (see Decision 24).

The most striking contrast is in **active_computation** (mapped from TEST): Zendo traces devote {dist_rows.loc['TEST', 'pct']:.1f}% to evidence extraction vs 32.7% for math. This reflects the visual inductive reasoning domain, where the model must systematically check features across multiple panels. Conversely, **fact_retrieval** (mapped from DESCRIBE) is lower in Zendo (8.4% vs 20.1%), since visual scanning is more interleaved with hypothesis testing than factual recall in math.

**plan_generation** (HYPO + PLAN) is higher in Zendo (21.5% vs 15.5%), reflecting the centrality of hypothesis generation to the Zendo task structure. **uncertainty_mgmt + self_checking** is comparable across domains ({bogdan_um_zendo*100:.1f}% vs {bogdan_um_bogdan*100:.1f}%).

![Domain comparison](figures/domain_comparison_bogdan.png)

---

## 4. Sequential Structure

### 4a. Transition Probabilities

The 9x9 bigram transition matrix reveals structured sequential patterns. The top 5 transitions by count are:

| From | To | Count | P(next \\| current) |
|------|-----|-------|-------------------|
| TEST | TEST | {int(top5.iloc[0]['count']):,} | {top5.iloc[0]['probability']:.3f} |
| HYPO | TEST | {int(top5.iloc[1]['count']):,} | {top5.iloc[1]['probability']:.3f} |
| TEST | JUDGE | {int(top5.iloc[2]['count']):,} | {top5.iloc[2]['probability']:.3f} |
| JUDGE | HYPO | {int(top5.iloc[3]['count']):,} | {top5.iloc[3]['probability']:.3f} |
| DESCRIBE | DESCRIBE | {int(top5.iloc[4]['count']):,} | {top5.iloc[4]['probability']:.3f} |

The dominant **HYPO->TEST->JUDGE->HYPO** cycle is clearly visible: hypotheses lead to evidence testing (P=0.628), testing leads to judgement (P=0.172) or more testing (P=0.692), and judgement leads back to a new hypothesis (P=0.662).

Self-transition rates vary substantially: TEST ({st.loc['TEST', 'self_transition_rate']:.3f}) and DESCRIBE ({st.loc['DESCRIBE', 'self_transition_rate']:.3f}) show high self-transition, reflecting extended runs of evidence extraction and panel scanning. JUDGE has low self-transition ({st.loc['JUDGE', 'self_transition_rate']:.3f}), consistent with its role as a punctual verdict rather than an extended activity.

![Transition matrix](figures/transition_matrix_micro.png)

### 4b. Transition Entropy

Per-trace transition entropy (Shannon entropy of bigram distribution) measures the diversity of sequential reasoning patterns. Overall mean entropy is **{ent_overall['mean']:.3f} bits** (SD = {ent_overall['sd']:.3f}).

Entropy varies significantly by task (Kruskal-Wallis H = 10.738, p = 0.013), with Task 1 showing the highest entropy (M = {entropy_stats[(entropy_stats['group'] == 'task1') & (entropy_stats['metric'] == 'transition_entropy')].iloc[0]['mean']:.3f}) and Task 3 the lowest (M = {entropy_stats[(entropy_stats['group'] == 'task3') & (entropy_stats['metric'] == 'transition_entropy')].iloc[0]['mean']:.3f}). This suggests that conjunctive rules (Task 3) elicit more stereotyped reasoning patterns.

Completed traces show significantly higher transition entropy than truncated traces (M = {ent_completed['mean']:.3f} vs {ent_truncated['mean']:.3f}; Mann-Whitney U = 16,902.5, p < 0.001). This indicates that successful reasoning involves more diverse sequential patterns, while truncated traces show more repetitive cycling.

![Transition entropy distribution](figures/transition_entropy_histogram.png)

![Entropy by task](figures/transition_entropy_by_task.png)

![Entropy by completion](figures/transition_entropy_by_completion.png)

### 4c. Sequence Characteristics

All {total_traces} traces open with **{opening_mode}**, confirming that the model consistently begins by restating the task. Closing labels differ sharply by completion status: {closing_counts.iloc[0]} traces ({closing_counts.iloc[0]/total_traces*100:.1f}%) end with {closing_counts.index[0]}, followed by {closing_counts.iloc[1]} ending with {closing_counts.index[1]}.

The first HYPO sentence appears at a mean normalised position of **{mean_first_hypo:.3f}** (median {median_first_hypo:.3f}), with {n_with_hypo}/{total_traces} traces containing at least one HYPO. The early median indicates that most traces transition quickly from scanning to hypothesis testing.

**Phase structure:** Traces decompose into three phases: scanning (before first HYPO), cycling (first HYPO to last JUDGE), and convergence (after last JUDGE). On average, scanning occupies {mean_scanning*100:.1f}% of the trace, cycling {mean_cycling*100:.1f}%, and convergence {mean_convergence*100:.1f}%. The cycling phase dominates, reflecting the iterative nature of hypothesis-testing reasoning. Completed traces show more scanning ({comp_scanning*100:.1f}%) and convergence ({comp_convergence*100:.1f}%) than truncated traces ({trunc_scanning*100:.1f}% and {trunc_convergence*100:.1f}%), which are almost entirely in the cycling phase ({trunc_cycling*100:.1f}%).

**Reasoning strategies:** The corpus exhibits three distinct reasoning strategies: full_cycling ({n_full_cycling} traces) with the standard HYPO->TEST->JUDGE pattern, scan_test_conclude ({n_scan_test} traces) where hypotheses are proposed but never formally judged, and direct_insight ({n_direct} traces) where the model reaches a conclusion without explicit hypothesis generation. The 17 non-cycling traces all completed successfully, representing alternative reasoning pathways rather than failures.

![Opening categories](figures/opening_categories.png)

![Closing categories](figures/closing_categories.png)

![First HYPO position](figures/first_hypo_position.png)

![Phase structure](figures/phase_structure.png)

---

## 5. Dependency Architecture

### 5a. Dependency Coverage and Distance

{r['dependency_coverage_pct']}% of sentences have at least one dependency edge (`depends_on`). The primary exception is ORIENT (51.4% coverage), which typically has no antecedents. Mean dependencies per sentence is {r['mean_deps_per_sentence']:.2f}.

Dependency distances (gap in sentence IDs between a sentence and its dependency) vary by category: HYPO shows the longest mean distance ({dep_stats.set_index('label').loc['HYPO', 'mean_distance']:.1f} sentences), reflecting hypotheses that reference earlier scanning or prior cycles. TEST and JUDGE have shorter mean distances ({dep_stats.set_index('label').loc['TEST', 'mean_distance']:.1f} and {dep_stats.set_index('label').loc['JUDGE', 'mean_distance']:.1f}), as they typically reference nearby sentences within the current hypothesis cycle.

![Dependency distance histogram](figures/dependency_distance_histogram.png)

![Dependency distance by label](figures/dependency_distance_by_label.png)

### 5b. Degree Analysis by Category

In-degree (how often a sentence is referenced) and out-degree (how many dependencies a sentence declares) reveal the informational roles of each category:

- **{highest_in}** has the highest mean in-degree ({deg.loc[highest_in, 'mean_in_degree']:.3f}), meaning sentences of this type are most frequently referenced by downstream reasoning.
- **{highest_out}** has the highest mean out-degree ({deg.loc[highest_out, 'mean_out_degree']:.3f}), meaning sentences of this type declare the most dependencies.
- **RULE** shows high in-degree ({deg.loc['RULE', 'mean_in_degree']:.3f}) but near-zero out-degree ({deg.loc['RULE', 'mean_out_degree']:.3f}), consistent with its role as a terminal conclusion that references prior reasoning but is rarely referenced itself.
- **ORIENT** has low in-degree ({deg.loc['ORIENT', 'mean_in_degree']:.3f}), confirming that opening sentences are rarely relevant to downstream hypothesis testing.

![Degree by category](figures/dependency_degree_by_category.png)

### 5c. Graph-Level Statistics

Per-trace dependency graphs have a mean density of **{gs_density['mean']:.4f}** (SD = {gs_density['sd']:.4f}), confirming sparse connectivity dominated by local edges. The mean longest directed path is **{gs_lp['mean']:.1f}** sentences (SD = {gs_lp['sd']:.1f}), and traces have a mean of **{gs_wcc['mean']:.1f}** weakly connected components (median {gs_wcc['median']:.0f}).

Graph density varies significantly by task (Kruskal-Wallis H = 30.797, p < 0.001). Completed traces have higher density (M = {gs_dens_comp['mean']:.4f}) than truncated traces (M = {gs_dens_trunc['mean']:.4f}), reflecting their shorter length and tighter dependency structure.

Longest directed path differs significantly by completion status (Mann-Whitney U = 3,421.5, p < 0.001): truncated traces have longer critical paths (M = {gs_lp_trunc['mean']:.1f}) than completed traces (M = {gs_lp_comp['mean']:.1f}). This reflects trace length: truncated traces tend to be longer (more sentences before hitting the token limit), producing longer dependency chains without converging.

Only 1 out of {total_traces} traces contained a cyclic dependency graph (an auto-coding artefact), handled by computing longest path on the graph's DAG condensation.

![Graph density by task](figures/graph_density_by_task.png)

![Longest path by completion](figures/longest_path_by_completion.png)

![Longest path vs length](figures/longest_path_vs_length.png)

---

## 6. Hypothesis Dynamics

### 6a. Hypothesis Cycles

Hypothesis cycles (HYPO->TEST*->JUDGE sequences identified via the dependency graph) are the fundamental unit of hypothesis-testing activity. The corpus contains **{n_cycles:,} cycles** across {total_traces} traces, with a mean of **{mean_cycles_per_trace:.1f} cycles per trace** and a mean cycle length of **{mean_cycle_len:.1f} sentences**. The cycle completion rate (proportion of HYPOs that reach a dependency-linked JUDGE) is **{cycle_completion_rate}%**, meaning roughly half of hypotheses are abandoned before a formal verdict.

![Cycle length distribution](figures/cycle_length_distribution.png)

### 6b. Hypothesis Recycling and Perseveration

Hypothesis status (computed post-hoc from semantic similarity) reveals that only **{r['pct_hypo_novel']}%** of hypotheses are genuinely novel. The majority are either **revised** ({r['pct_hypo_revised']}%, modified from a prior hypothesis) or **repeated** ({r['pct_hypo_repeated']}%, restated without meaningful change). This recycling is pervasive across all four tasks.

**{r['pct_perseverative_traces']}%** of traces are classified as **perseverative** (containing at least one repetition chain of length >= 5), indicating that nearly half of all traces include extended sequences where the model restates the same hypothesis multiple times without productive revision.

Critically, **{100 - r['pct_final_rule_from_novel']}%** of final rules in completed traces derive from revised or repeated hypotheses rather than novel ones. This suggests that hypothesis recycling is not purely pathological — it may serve a consolidation function, with the model progressively refining its answer through iterative restatement.

![Hypothesis status by task](figures/hypo_status_by_task.png)

![Repetition by position](figures/repetition_by_position.png)

---

## 7. Summary of Key Findings

1. **TEST dominates the reasoning corpus** at {dist_rows.loc['TEST', 'pct']:.1f}% of sentences, with the INVESTIGATE macro-category (HYPO + TEST + JUDGE) accounting for {dist_rows.loc['HYPO', 'pct'] + dist_rows.loc['TEST', 'pct'] + dist_rows.loc['JUDGE', 'pct']:.1f}% overall. Visual inductive reasoning is fundamentally an evidence-extraction task.

2. **Coding reliability is strong** (mean kappa = {mean_k_micro:.3f} micro, {mean_k_macro:.3f} macro), with 8/9 categories passing the >= 0.50 gate. SYNTHESIZE is the only weak category.

3. **The HYPO->TEST->JUDGE->HYPO cycle** is the dominant sequential motif, visible in both the transition matrix and the dependency graph structure. Traces contain a mean of {mean_cycles_per_trace:.1f} such cycles.

4. **Completed traces show more diverse reasoning** (higher transition entropy, M = {ent_completed['mean']:.3f} vs {ent_truncated['mean']:.3f} for truncated, p < 0.001) and tighter dependency structure (higher graph density, shorter longest paths).

5. **Hypothesis recycling is pervasive**: only {r['pct_hypo_novel']}% of hypotheses are novel, {r['pct_perseverative_traces']}% of traces are perseverative, yet {100 - r['pct_final_rule_from_novel']}% of final rules derive from recycled hypotheses.

6. **Phase structure is dominated by cycling** ({mean_cycling*100:.1f}% of trace length), with brief scanning ({mean_scanning*100:.1f}%) and minimal convergence ({mean_convergence*100:.1f}%). Three reasoning strategies emerge: full_cycling ({n_full_cycling}), scan_test_conclude ({n_scan_test}), and direct_insight ({n_direct}).

7. **Dependency graphs are sparse but structured**: mean density {gs_density['mean']:.4f}, with {highest_in} showing the highest in-degree (most referenced) and {highest_out} the highest out-degree (most dependencies declared).

8. **Domain comparison with Bogdan et al.** shows Zendo reasoning is more evidence-extraction-heavy (50.2% vs 32.7% active_computation) and more hypothesis-driven (21.5% vs 15.5% plan_generation) than math problem-solving. Uncertainty-related categories are comparable ({bogdan_um_zendo*100:.1f}% vs {bogdan_um_bogdan*100:.1f}%).

9. **Task 3 (conjunctive rules)** shows the most stereotyped reasoning (lowest transition entropy) and longest cycling phase, consistent with its higher complexity.

10. **Cycle completion rate is moderate** ({cycle_completion_rate}%): roughly half of hypotheses are abandoned before reaching a JUDGE verdict, suggesting that the model frequently shifts strategy mid-evaluation.

---

## 8. Implications for Study 2 (Mechanistic Verification)

The corpus characterisation identifies several targets for mechanistic probing in Study 2:

- **TEST as primary probe target.** TEST's dominance ({dist_rows.loc['TEST', 'pct']:.1f}% of corpus) makes it the richest source of activation data for training linear probes. The category's high self-transition rate ({st.loc['TEST', 'self_transition_rate']:.3f}) and moderate dependency distances suggest that TEST activations should show stable, locally coherent patterns.

- **HYPO->TEST->JUDGE as a sequential signature.** The dominant hypothesis-testing cycle should produce detectable activation trajectories. Probes trained on transition points (HYPO->TEST, TEST->JUDGE) may reveal how the model internally represents the shift from hypothesis generation to evidence evaluation to verdict.

- **MONITOR as phase transition marker.** MONITOR sentences (2.3% of corpus) mark moments of meta-cognitive reflection. Their position at cycling-phase boundaries makes them candidates for steering vector extraction — if MONITOR has a mechanistic signature, steering toward it could modulate the model's tendency to reflect vs perseverate.

- **Perseveration as a mechanistic target.** {r['pct_perseverative_traces']}% of traces are perseverative. If "stuck" traces are mechanistically distinguishable from productive exploration (e.g., different activation trajectories during repeated vs revised hypotheses), this would provide evidence for faithfulness — the model's internal state reflects whether its reasoning is productive.

- **Reasoning strategy as a grouping variable.** The three reasoning strategies (full_cycling, scan_test_conclude, direct_insight) provide a natural grouping for Study 2 analyses — mechanistic signatures may differ across strategies.

- **The SYNTHESIZE reliability concern.** SYNTHESIZE's low validation kappa means probe training data for this category may contain noise. Analyses involving SYNTHESIZE should include sensitivity checks or consider merging with DESCRIBE at the macro level.

- **Graph structure as a probe feature.** Graph density and longest-path length characterise traces at the structural level. If activation patterns correlate with these graph features (e.g., denser graphs produce more coherent activation trajectories), this would link sequential reasoning structure to the model's internal representations.

---

*Report compiled by `notebooks/study1_04_report.ipynb` from outputs of Phase 4 analysis notebooks.*
""")

report_text = "\n".join(sections)
print(f'Report assembled: {len(report_text)} chars, ~{len(report_text.split())} words')

Report assembled: 18956 chars, ~2494 words


In [6]:
# ── Save report ──────────────────────────────────────────────────────────────

report_path = OUTPUT / 'study1_report.md'
report_path.write_text(report_text, encoding='utf-8')
print(f"Report saved: {report_path}")
print(f"Length: {len(report_text)} characters, ~{len(report_text.split())} words")

Report saved: outputs\study1_analysis\study1_report.md
Length: 18956 characters, ~2494 words
